# 레이블 전파 (Label Propagation)

GCN이 이웃의 **특징**을 모으고 가중치를 학습한다면, Label Propagation은 이미 알려진 노드의 **레이블 정보**를 그래프의 이웃으로 반복 전달합니다.

원본과 동일하게 PyTorch Geometric(PyG)의 `KarateClub()`과 기본 `train_mask`를 사용합니다. 네 커뮤니티에서 하나씩, 총 4개 노드의 레이블만 알려져 있다고 가정합니다. 전체 그래프의 연결을 사용하여 나머지 30개 노드의 클래스를 추론하는 **(Semi-supervised & Transductive) node classification**입니다.

## 1. Karate Club 데이터 불러오기

In [1]:
from torch_geometric.datasets import KarateClub

dataset = KarateClub()

data = dataset[0]
train_mask = data.train_mask.clone()
test_mask = ~train_mask

c:\Users\KETI\anaconda3\envs\ai_tutorial\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Label Propagation 구현하기

각 노드에 클래스별 점수 4개를 둡니다. 초기 행렬 $Y^{(0)}\in\mathbb{R}^{34\times4}$는 알려진 노드에서만 원-핫 레이블이고, 나머지 행은 모두 0입니다.

$$
S=D^{-1/2}AD^{-1/2},\qquad
Y^{(t+1)}=\operatorname{clip}_{[0,1]}\left(\alpha S Y^{(t)}+(1-\alpha)Y^{(0)}\right).
$$

$A$는 인접 행렬, $D$는 차수 대각 행렬입니다. $\alpha=0.9$이면 이웃에서 전파된 점수에 0.9를 곱하고 초기 레이블 정보에 0.1을 곱해 더합니다. 

In [40]:
from torch_geometric.nn.models import LabelPropagation
import torch
import torch.nn.functional as F

y0 = torch.zeros((data.num_nodes, dataset.num_classes), dtype=torch.float32)
y0[train_mask] = F.one_hot(
    data.y[train_mask], num_classes=dataset.num_classes
).float()

pyg_model = LabelPropagation(num_layers=100, alpha=0.9)
y_hat = pyg_model(y0, data.edge_index)

# pred = y_hat.argmax(dim=1)
# test_accuracy = (pred[test_mask] == data.y[test_mask]).float().mean().item()
test_accuracy = (
        F.softmax(y_hat[test_mask], dim=1).argmax(dim=1) == data.y[test_mask]
    ).float().mean().item()
print(f'Test accuracy (30 nodes): {test_accuracy:.2%}')

Test accuracy (30 nodes): 70.00%
